In [ ]:
# --- setup --------------------------------------------------------
import sys
sys.path.insert(0, r"D:\Projects\done\eth_bot\eth_bot_v2")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
from pathlib import Path

DB = Path(r"D:\Projects\done\eth_bot\eth_bot_v2\database\market.db")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
plt.rcParams["figure.figsize"] = (12, 4)

def sql(q, **p):
    with sqlite3.connect(DB) as c:
        return pd.read_sql_query(q, c, params=p)

from analysis.loader import load_ohlcv
from analysis.indicators import add_all
from analysis import stats


In [ ]:
# --- load ---------------------------------------------------------
fg = sql("SELECT ts, value, label FROM fear_greed ORDER BY ts")
fg["date"] = pd.to_datetime(fg.ts, unit="ms")
fg = fg.set_index("date")
fg.tail()

In [ ]:
# --- F&G over time -----------------------------------------------
fg.value.plot(title="Fear & Greed")
plt.axhline(50, ls="--", lw=0.7, color="gray")
plt.tight_layout()
plt.show()

In [ ]:
# --- F&G vs price -------------------------------------------------
px = load_ohlcv().close
fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
px.plot(ax=ax[0], title="close")
fg.value.plot(ax=ax[1], color="purple", title="fear & greed")
plt.tight_layout()
plt.show()

In [ ]:
# --- correlation: today's F&G vs next-day return -----------------
px_r = load_ohlcv().close.pct_change().shift(-1)  # next-day return
merged = pd.concat([fg.value, px_r], axis=1).dropna()
merged.columns = ["fng", "next_ret"]
merged.corr()